In [2]:
import json
import logging
import os
from datetime import datetime, UTC
from pathlib import Path

import pandas as pd
import vk_api
from dotenv import load_dotenv

from Data.features.VKFeatureExtractor import VKFeaturesExtractor

load_dotenv("/Users/olgagavrilenko/DataspellProjects/Course-work/Scripts/token.env")
logging.basicConfig(level=logging.INFO)

INPUT_JSON = Path("Passings 18.12.2024 (4).json")
OUTPUT_DIR = Path("prepared_from_vk")

TARGET_MAP = {
    "result/0/Экстраверсия–интроверсия": "Экстраверсия–интроверсия",
    "result/0/Привязанность–обособленность": "Привязанность–обособленность",
    "result/0/Самоконтроль–импульсивность": "Самоконтроль–импульсивность",
    "result/0/Эмоциональная_устойчивость–эмоциональная_неустойчивость": "Эмоциональная_устойчивость–эмоциональная_неустойчивость",
    "result/0/Экспрессивность–практичность": "Экспрессивность–практичность",
}

TARGETS_FILE = OUTPUT_DIR / "targets_ready.csv"
NODE_FEATURES_FILE = OUTPUT_DIR / "node_features_ready.csv"
GRAPH_CONTEXT_FILE = OUTPUT_DIR / "user_graph_context.csv"
PROCESSED_IDS_FILE = OUTPUT_DIR / "processed_vk_ids.csv"
INVALID_IDS_FILE = OUTPUT_DIR / "invalid_vk_ids.csv"
ERROR_IDS_FILE = OUTPUT_DIR / "error_vk_ids.csv"
FILTERED_TARGETS_FILE = OUTPUT_DIR / "targets_filtered.csv"


# ============================================
# ОБЩИЕ УТИЛИТЫ
# ============================================
def ensure_token():
    token = os.getenv("ACCESS_TOKEN_VK")
    if not token:
        raise ValueError("ACCESS_TOKEN_VK не найден в .env")
    return token


def append_row_to_csv(row: dict, path: Path):
    df_row = pd.DataFrame([row])
    write_header = not path.exists() or path.stat().st_size == 0
    df_row.to_csv(path, mode="a", header=write_header, index=False)


def read_existing_ids(path: Path, id_col: str = "vk_id") -> set[int]:
    if not path.exists():
        return set()

    try:
        df = pd.read_csv(path)
        if id_col not in df.columns:
            return set()
        ids = pd.to_numeric(df[id_col], errors="coerce").dropna().astype(int).tolist()
        return set(ids)
    except Exception:
        return set()

def load_raw_json_records(path: str | Path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Файл не найден: {path.resolve()}")

    text = path.read_text(encoding="utf-8", errors="replace").strip()
    data = json.loads(text)

    if isinstance(data, list):
        print(f"Прочитан JSON-массив: {len(data)} записей")
        return data
    if isinstance(data, dict):
        print("Прочитан одиночный JSON-объект")
        return [data]

    raise ValueError("Неожиданный формат JSON")


def extract_targets_from_record(record: dict, target_map: dict[str, str]) -> dict | None:
    vk_id = record.get("vk_id")
    completion_date = record.get("completion_date")
    result = record.get("result")

    if vk_id is None or result is None:
        return None

    if isinstance(result, str):
        try:
            result = json.loads(result)
        except Exception:
            return None

    if not isinstance(result, list) or len(result) == 0:
        return None

    first_block = result[0]
    if not isinstance(first_block, dict):
        return None

    row = {
        "vk_id": vk_id,
        "completion_date": completion_date,
    }

    for out_col, source_key in target_map.items():
        row[out_col] = first_block.get(source_key)

    return row


def load_targets_from_json(path: str | Path, target_map: dict[str, str]) -> pd.DataFrame:
    records = load_raw_json_records(path)

    rows = []
    for rec in records:
        row = extract_targets_from_record(rec, target_map)
        if row is not None:
            rows.append(row)

    print(f"После извлечения таргетов: {len(rows)} записей")

    if not rows:
        raise ValueError("Не удалось извлечь ни одной записи с таргетами.")

    df_targets = pd.DataFrame(rows)

    df_targets["completion_date"] = pd.to_datetime(
        df_targets["completion_date"],
        errors="coerce"
    )

    target_cols = list(target_map.keys())

    df_targets = df_targets.dropna(subset=["vk_id"]).copy()
    df_targets = df_targets.dropna(subset=target_cols, how="all").copy()

    # Как в репозитории: оставить последнюю запись по completion_date
    df_targets = df_targets.sort_values("completion_date")
    df_targets = df_targets.drop_duplicates(subset=["vk_id"], keep="last").copy()
    df_targets = df_targets.drop(columns=["completion_date"])

    df_targets["vk_id"] = pd.to_numeric(df_targets["vk_id"], errors="coerce")
    df_targets = df_targets.dropna(subset=["vk_id"]).copy()
    df_targets["vk_id"] = df_targets["vk_id"].astype(int)

    print(f"После deduplicate по vk_id: {len(df_targets)} пользователей")
    return df_targets


def profile_is_accessible(vk, user_id: int) -> tuple[bool, str]:
    try:
        response = vk.users.get(
            user_ids=str(user_id),
            fields="is_closed,can_access_closed"
        )

        if not response:
            return False, "empty_response"

        user = response[0]

        if user.get("deactivated"):
            return False, f"deactivated:{user.get('deactivated')}"

        if user.get("is_closed") and not user.get("can_access_closed", False):
            return False, "closed_profile"

        return True, "ok"

    except vk_api.exceptions.ApiError as e:
        if getattr(e, "code", None) in (15, 18, 30):
            return False, f"api_error:{e.code}"
        raise


def collect_one_user_features(extractor: VKFeaturesExtractor, user_id: int) -> dict | None:
    features = extractor._extract_for_user(user_id)

    # если вернулся только user_id без признаков
    if not features or len(features.keys()) <= 1:
        return None

    rename_map = {
        "user_id": "vk_id",
        "male_count": "male_friends",
        "female_count": "female_friends",
        "unknown_count": "unknown_friends",
    }

    row = {rename_map.get(k, k): v for k, v in features.items()}

    wanted_order = [
        "vk_id",
        "friends_count",
        "male_friends",
        "female_friends",
        "unknown_friends",
        "photo_count",
        "likes_total",
        "average_likes",
        "groups_count",
        "average_member",
    ]

    clean_row = {}
    for col in wanted_order:
        clean_row[col] = row.get(col, None)

    return clean_row

def collect_one_user_graph_context(
        extractor: VKFeaturesExtractor,
        user_id: int,
        valid_vk_id_set: set[int],
) -> dict:
    row = {
        "vk_id": user_id,
        "friends_json": "[]",
        "groups_json": "[]",
        "city_id": None,
        "university": None,
        "faculty": None,
    }

    # friends
    try:
        friends_resp = extractor._get_friends(user_id)
        friends_items = friends_resp.get("items", [])
        friends_ids = [
            f["id"] for f in friends_items
            if f.get("id") in valid_vk_id_set
        ]
        row["friends_json"] = json.dumps(sorted(set(friends_ids)), ensure_ascii=False)
    except Exception as e:
        logging.warning(f"friends failed for {user_id}: {e}")

    # groups
    try:
        groups_resp = extractor._get_groups(user_id)
        groups_items = groups_resp.get("items", [])
        group_ids = [g["id"] for g in groups_items if g.get("id") is not None]
        row["groups_json"] = json.dumps(sorted(set(group_ids)), ensure_ascii=False)
    except Exception as e:
        logging.warning(f"groups failed for {user_id}: {e}")

    # city
    try:
        city_resp = extractor._get_mutual_cities([user_id])
        if city_resp and isinstance(city_resp, list):
            row["city_id"] = city_resp[0].get("city", {}).get("id")
    except Exception as e:
        logging.warning(f"city failed for {user_id}: {e}")

    # education
    try:
        edu_resp = extractor._get_education([user_id])
        if edu_resp and isinstance(edu_resp, list):
            row["university"] = edu_resp[0].get("university")
            row["faculty"] = edu_resp[0].get("faculty")
    except Exception as e:
        logging.warning(f"education failed for {user_id}: {e}")

    return row

def save_filtered_targets(df_targets: pd.DataFrame):
    processed_ids = read_existing_ids(PROCESSED_IDS_FILE, id_col="vk_id")
    if not processed_ids:
        return

    df = df_targets.copy()
    df = df[df["vk_id"].isin(processed_ids)].copy()
    df.to_csv(FILTERED_TARGETS_FILE, index=False)

def main():
    ensure_token()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print("1. Читаю JSON и извлекаю таргеты...")
    df_targets = load_targets_from_json(INPUT_JSON, TARGET_MAP)
    df_targets.to_csv(TARGETS_FILE, index=False)

    all_vk_ids = df_targets["vk_id"].drop_duplicates().tolist()

    processed_ids = read_existing_ids(PROCESSED_IDS_FILE, id_col="vk_id")
    invalid_ids = read_existing_ids(INVALID_IDS_FILE, id_col="vk_id")

    done_ids = processed_ids | invalid_ids
    remaining_ids = [uid for uid in all_vk_ids if uid not in done_ids]

    print(f"Всего пользователей в таргетах: {len(all_vk_ids)}")
    print(f"Уже полностью обработано: {len(processed_ids)}")
    print(f"Уже invalid: {len(invalid_ids)}")
    print(f"Осталось обработать: {len(remaining_ids)}")

    if not remaining_ids:
        print("Новых пользователей для обработки нет.")
        save_filtered_targets(df_targets)
        print(TARGETS_FILE)
        print(NODE_FEATURES_FILE)
        print(GRAPH_CONTEXT_FILE)
        print(FILTERED_TARGETS_FILE)
        return

    extractor = VKFeaturesExtractor(users_id=remaining_ids)
    vk = extractor.vk
    valid_vk_id_set = set(all_vk_ids)

    processed_now = 0

    try:
        for idx, user_id in enumerate(remaining_ids, start=1):
            if idx % 25 == 0 or idx == 1:
                print(f"[{idx}/{len(remaining_ids)}] Обрабатываю vk_id={user_id}")

            try:
                accessible, reason = profile_is_accessible(vk, user_id)

                if not accessible:
                    append_row_to_csv(
                        {
                            "vk_id": user_id,
                            "reason": reason,
                            "checked_at": datetime.now(UTC).isoformat(),
                        },
                        INVALID_IDS_FILE,
                    )
                    continue

                # 1) обычные признаки
                node_row = collect_one_user_features(extractor, user_id)
                if node_row is None:
                    append_row_to_csv(
                        {
                            "vk_id": user_id,
                            "reason": "feature_extraction_failed",
                            "checked_at": datetime.now(UTC).isoformat(),
                        },
                        ERROR_IDS_FILE,
                    )
                    continue

                # 2) признаки для графа
                graph_row = collect_one_user_graph_context(
                    extractor=extractor,
                    user_id=user_id,
                    valid_vk_id_set=valid_vk_id_set,
                )

                # Сохраняем оба файла
                append_row_to_csv(node_row, NODE_FEATURES_FILE)
                append_row_to_csv(graph_row, GRAPH_CONTEXT_FILE)

                # И только после успешной записи обоих — помечаем id как обработанный
                append_row_to_csv(
                    {
                        "vk_id": user_id,
                        "processed_at": datetime.now(UTC).isoformat(),
                    },
                    PROCESSED_IDS_FILE,
                )

                processed_now += 1

            except KeyboardInterrupt:
                print("Остановлено пользователем. Уже сохранённые данные не потеряются.")
                raise

            except Exception as e:
                logging.exception(f"Ошибка на user_id={user_id}: {e}")
                append_row_to_csv(
                    {
                        "vk_id": user_id,
                        "reason": f"exception:{type(e).__name__}",
                        "checked_at": datetime.now(UTC).isoformat(),
                    },
                    ERROR_IDS_FILE,
                )
                continue

    finally:
        save_filtered_targets(df_targets)

    print("Готово.")
    print(f"Новых успешно обработанных пользователей в этом запуске: {processed_now}")
    print(TARGETS_FILE)
    print(NODE_FEATURES_FILE)
    print(GRAPH_CONTEXT_FILE)
    print(PROCESSED_IDS_FILE)
    print(INVALID_IDS_FILE)
    print(ERROR_IDS_FILE)
    print(FILTERED_TARGETS_FILE)


if __name__ == "__main__":
    main()

1. Читаю JSON и извлекаю таргеты...
Прочитан JSON-массив: 7150 записей
После извлечения таргетов: 7150 записей
После deduplicate по vk_id: 6987 пользователей
Всего пользователей в таргетах: 6987
Уже полностью обработано: 2
Уже invalid: 1
Осталось обработать: 6984
[1/6984] Обрабатываю vk_id=237338911
[25/6984] Обрабатываю vk_id=202006292
[50/6984] Обрабатываю vk_id=398537232
[75/6984] Обрабатываю vk_id=643440686
[100/6984] Обрабатываю vk_id=347245137
[125/6984] Обрабатываю vk_id=690526628
[150/6984] Обрабатываю vk_id=154964756
[175/6984] Обрабатываю vk_id=691634118
[200/6984] Обрабатываю vk_id=107570655
[225/6984] Обрабатываю vk_id=37219332
[250/6984] Обрабатываю vk_id=159221004
[275/6984] Обрабатываю vk_id=38736521
[300/6984] Обрабатываю vk_id=342677599
Остановлено пользователем. Уже сохранённые данные не потеряются.


KeyboardInterrupt: 